## Part 2, Stage A, Step 1: Today's system (PostgreSQL + polling detector)

### Predictions

| Question | Prediction | Reasoning |
|---|---|---|
| Does the detector keep up with 20 machines (200 readings/s)? | Yes, backlog stays near 0 | 200 × 3 ms = 0.6 s of work per second. |
| Does the detector read as many rows as the table holds? | No, fewer | Some rows are never seen by the detector. |
| Does `missing` grow? | Yes | `missing` counts gaps in `seq`, which are the skipped rows. |
| Are the missing readings in the table? | Yes | They were inserted and committed, the detector just never read them. |
| Why were they skipped? | The `id` is assigned at INSERT, but a row is visible to others only after COMMIT. The detector remembers the highest id it saw, so rows committed later with a lower id are skipped. | Timeline: gateway 1 gets ids 100–114 but commits later, gateway 2 gets ids 115–129 and commits at once. A poll in between sees only 115–129 and sets `last_id = 129`. |

### Observation

Terminal 1: `python baseline_detector.py` (started first). Terminal 2: `python plant.py --sink postgres --machines 20`.

Plant output:

```
run 1794387786: 20 machines x 10 Hz = 200 readings/s -> postgres (Ctrl+C to stop)
  t=    5s      1,020 readings sent (203/s)
  t=   10s      2,020 readings sent (202/s)
  ...
  t=   35s      7,020 readings sent (201/s)
  t=   40s  fault starts on M-006 (repaired in 45 s)
  t=   40s      8,020 readings sent (200/s)  faulty: M-006
  ...
  t=   80s     16,020 readings sent (200/s)  faulty: M-006
  t=   85s     17,020 readings sent (200/s)  faulty: M-006
  t=   90s     18,020 readings sent (200/s)
run 1794387786: sent 18,120 readings in 91 s
```

Detector output (status lines, `ANOMALY` lines shortened):

```
[baseline] polling table readings for id > 0 (Ctrl+C to stop)
[baseline] read      204  backlog       0 rows  delay   0.5 s  missing      5  anomalies 0
[baseline] read    1,101  backlog     170 rows  delay   1.2 s  missing     49  anomalies 0
[baseline] read    2,181  backlog      14 rows  delay   0.2 s  missing     94  anomalies 0
[baseline] read    3,033  backlog      20 rows  delay   0.3 s  missing    227  anomalies 0
[baseline] read    3,860  backlog      14 rows  delay   0.2 s  missing    418  anomalies 0
[baseline] read    4,734  backlog      40 rows  delay   0.4 s  missing    546  anomalies 0
[baseline] read    5,542  backlog      30 rows  delay   0.3 s  missing    708  anomalies 0
[baseline] read    6,492  backlog      31 rows  delay   0.3 s  missing    813  anomalies 0
[baseline] read    7,382  backlog       9 rows  delay   0.2 s  missing    928  anomalies 0
[baseline] read    8,226  backlog      29 rows  delay   0.3 s  missing  1,094  anomalies 0
[baseline] ANOMALY M-006 seq=487 score=5.03
[baseline] ANOMALY M-006 seq=488 score=5.3
... (many more ANOMALY lines for M-006, scores rising to about 23) ...
[baseline] read    9,125  backlog      29 rows  delay   0.3 s  missing  1,195  anomalies 28
[baseline] read    9,900  backlog      12 rows  delay   0.2 s  missing  1,363  anomalies 73
[baseline] read   10,734  backlog      20 rows  delay   0.2 s  missing  1,540  anomalies 111
[baseline] read   11,564  backlog       0 rows  delay   0.2 s  missing  1,711  anomalies 155
[baseline] read   12,409  backlog      31 rows  delay   0.3 s  missing  1,851  anomalies 201
[baseline] read   13,301  backlog      27 rows  delay   0.3 s  missing  1,949  anomalies 245
[baseline] read   13,832  backlog     680 rows  delay   3.6 s  missing  1,958  anomalies 273
[baseline] read   15,052  backlog     420 rows  delay   2.3 s  missing  1,973  anomalies 331
... (scores fall again from about 23 to 5 as the fault is repaired) ...
[baseline] ANOMALY M-006 seq=891 score=5.38
[baseline] read   16,124  backlog       0 rows  delay   0.0 s  missing  1,986  anomalies 370
[baseline] stopped: read 16,124 readings, last id 18,120, 1,986 readings missing, 370 anomalies
```

Row count in PostgreSQL (latest run):

Command: `docker exec -it postgres psql -U postgres -d factory -c "SELECT count(*) FROM readings WHERE run = (SELECT max(run) FROM readings);"`

```
 count 
-------
 18120
(1 row)
```

### Analysis

- **The detector keeps up:** backlog is mostly 0 to 40 rows and delay 0.2 to 0.5 s. The backlog briefly rose to 680 rows (delay 3.6 s) at the fault's peak and recovered afterwards.
- **The detector read fewer rows than the table holds:** 16,124 vs 18,120, so **1,996 readings (about 11%) were never read**. Its own `missing` counter says 1,986. The difference of 10 is probably the last skipped readings of each machine, which cannot be detected as gaps because no later reading follows them.
- **The missing readings are in the table.** The plant sent 18,120 readings and the table holds 18,120, so nothing was lost on the way in. The detector never saw them.
- **Cause:** the `id` (BIGSERIAL) is assigned at INSERT, but other sessions only see a row after COMMIT. With several gateways inserting concurrently, a row with a lower id can be committed after one with a higher id. If the detector polls in between, it sets `last_id` to the higher id and then asks for `id > last_id`, so the late rows are skipped forever.
- **Consequence:** requirement R2 (the model gets every reading of a machine, in `seq` order) is violated, even though the detector looks healthy (low backlog, low delay).

**Key idea:** polling with `WHERE id > last_id` is not safe when several writers insert concurrently, because id order is not commit order.

## Part 2, Stage A, Step 2: Full plant (60 machines, 600 readings/s)

### Predictions

| Question | Prediction | Reasoning |
|---|---|---|
| What happens to `backlog`? | Grows steadily, to roughly 15,000 to 16,000 rows after 60 s | 600 readings/s arrive, but one detector scores about 333/s (3 ms each). |
| What happens to `delay`? | Grows steadily, to about 45 s after 60 s | New readings wait behind the whole backlog. |
| What happens to `missing`? | Keeps growing, probably faster | The id-vs-commit race from Step 1 is still there, with more concurrent writers. |
| What happens after the plant stops? | The detector keeps working through its backlog | It must still score every waiting row. |

### Observation

Terminal 1: `python -u baseline_detector.py` (started first). Terminal 2: `python -u plant.py --sink postgres` (60 machines). Both were stopped after about 60 s, the detector right after the plant.

Detector output:

```
[baseline] polling table readings for id > 18120 (Ctrl+C to stop)
[baseline] read        0  backlog       0 rows  delay   0.0 s  missing      0  anomalies 0
[baseline] read        0  backlog       0 rows  delay   0.0 s  missing      0  anomalies 0
[baseline] read        0  backlog       0 rows  delay   0.0 s  missing      0  anomalies 0
[baseline] read        0  backlog       0 rows  delay   0.0 s  missing      0  anomalies 0
[baseline] read    1,304  backlog   2,356 rows  delay   4.3 s  missing     31  anomalies 0
[baseline] read    3,304  backlog   5,336 rows  delay   9.0 s  missing     76  anomalies 0
[baseline] read    5,304  backlog   8,661 rows  delay  14.6 s  missing     76  anomalies 0
[baseline] read    7,304  backlog  11,445 rows  delay  19.3 s  missing     76  anomalies 0
[baseline] read    9,304  backlog  14,005 rows  delay  23.5 s  missing     76  anomalies 0
[baseline] read   11,304  backlog  16,566 rows  delay  27.6 s  missing     76  anomalies 0
[baseline] read   13,304  backlog  19,245 rows  delay  32.3 s  missing     76  anomalies 0
[baseline] read   15,304  backlog  21,040 rows  delay  36.6 s  missing     76  anomalies 0
[baseline] stopped: read 15,304 readings, last id 34,221, 76 readings missing, 0 anomalies
```

Plant output:

```
run 1794388609: 60 machines x 10 Hz = 600 readings/s -> postgres (Ctrl+C to stop)
  t=    5s      3,060 readings sent (612/s)
  t=   10s      6,060 readings sent (606/s)
  t=   15s      9,060 readings sent (604/s)
  t=   20s     12,060 readings sent (602/s)
  t=   25s     15,060 readings sent (602/s)
  t=   30s     18,060 readings sent (602/s)
  t=   35s     21,060 readings sent (602/s)
  t=   40s  fault starts on M-029 (repaired in 45 s)
  t=   40s     24,060 readings sent (601/s)  faulty: M-029
  t=   45s     27,060 readings sent (601/s)  faulty: M-029
  t=   50s     30,060 readings sent (601/s)  faulty: M-029
  t=   55s     33,060 readings sent (601/s)  faulty: M-029
  t=   60s     36,060 readings sent (601/s)  faulty: M-029
run 1794388609: sent 36,420 readings in 61 s
```

Row count in PostgreSQL (latest run):

Command: `docker exec -it postgres psql -U postgres -d factory -c "SELECT count(*) FROM readings WHERE run = (SELECT max(run) FROM readings);"`

```
 count 
-------
 36420
(1 row)
```

### Analysis

| Prediction | Result |
|---|---|
| `backlog` grows steadily | Correct: 2,356 → 21,040 rows. My estimate (15,000 to 16,000) was too low. |
| `delay` grows steadily | Correct: 4.3 s → 36.6 s. My estimate (about 45 s) was a bit high. |
| `missing` keeps growing | **Wrong:** it rose to 76 early and then stayed at 76. |
| After the plant stops, the detector works through its backlog | Not observed, because the detector was stopped right after the plant. |

- **The detector cannot keep up:** it read 15,304 readings in about 61 s, which is about **250 readings/s**. That is below the theoretical 333/s (3 ms per reading), probably because polling and database overhead come on top. The plant sends about 600/s, so the backlog grows by roughly 350 rows per second.
- **The numbers add up:** 36,420 sent − 15,304 read = 21,116 unread rows, which matches the last backlog of 21,040.
- **The table is complete:** it holds 36,420 rows, exactly what the plant sent. Nothing was lost on the way in.
- **0 anomalies:** a fault started on M-029 at t=40 s, but at the end the detector was 36.6 s behind and was still working on readings from before the fault. A technician would receive no alert, which violates **R3** (anomalies within 5 s).
- **Why `missing` stopped growing (hypothesis, not verified):** the commit race only affects rows at the live end of the table. Once the detector had fallen behind, it read old rows that were long committed, so it no longer polled at the live edge. The race was not fixed, the detector was just too far behind for it to matter.

### Conclusion: two separate problems

1. **Throughput:** one detector instance is too slow for 600 readings/s, and 100,000 readings/s is far out of reach (R1, R3).
2. **Correctness:** polling with `WHERE id > last_id` can skip committed rows when the detector is at the live edge (R2).

**Key idea:** the database table as a hand-off point between gateways and detector is the problem. It breaks ordering and completeness, and it cannot be shared among parallel detectors without risking more missing or out-of-order readings.

## Part 2, Stage B: Decision record

### Context

CNC and injection-moulding machines send vibration readings. A stateful model (3 ms per reading) must detect bearing wear and raise an alert within 5 s (p95). Today: 60 machines × 10 Hz = 600 readings/s. Planned: 2,000 machines × 50 Hz = 100,000 readings/s. The raw readings must also feed an archive and a dashboard.

### Options

1. **Keep PostgreSQL, poll better** (several pollers, each for a part of the machines).
2. **Gateways write into Kafka** instead of PostgreSQL.

### Comparison with the requirements

| Requirement | Option 1: PostgreSQL | Option 2: Kafka |
|---|---|---|
| R1 Scale to 100,000 readings/s | At risk: very high insert rate plus many pollers on one database | Met: append-only log, scale with partitions and brokers |
| R2 Complete and ordered per machine | At risk: Stage A showed skipped rows (id order ≠ commit order) | Met: key = machine, so one machine stays in one partition, in order |
| R3 Fresh (≤ 5 s p95) | At risk: polling interval, commit race, load on the table | Met, if enough consumers keep the lag near 0 |
| R4 Independent consumers | Not met: every consumer adds queries to PostgreSQL | Met: each consumer group has its own offsets |

### Evidence from Stage A

- 20 machines (200 readings/s): the detector kept up, but 1,986 readings were never read although all 18,120 were in the table.
- 60 machines (600 readings/s): one detector fell 36.6 s behind (backlog 21,040 rows) and raised no alert during a fault.

### Decision

**Option 2: Kafka.** The gateways publish every reading with the machine ID as the key. The detector service runs as a consumer group, and the archive and dashboard each use their own group. PostgreSQL stays the system of record for the machine registry.

### Reasons

- Key → partition guarantees all readings of a machine reach the same detector in `seq` order (R2).
- One consumer per partition gives parallelism, so detectors can scale with the load (R1, R3).
- Independent groups let the archive and dashboard read without touching the detector or PostgreSQL (R4).
- Offsets and lag show whether the detector keeps up, and where it restarts after a failure.

### Numbers

| Load | Work per second of data | Detector instances at 3 ms/reading |
|---|---|---|
| 600 readings/s | 600 × 0.003 = 1.8 s | at least 2 (3 with headroom) |
| 100,000 readings/s | 100,000 × 0.003 = 300 s | at least 300 (about 400 with headroom) |

One instance handles at most 1 / 0.003 ≈ 333 readings/s. The Stage A baseline reached about 250/s including polling overhead. The topic therefore needs at least as many partitions as detector instances: a few now, several hundred at 100,000 readings/s.

### Tipping point

A small office with one reading per minute (about 0.017 readings/s) is served perfectly by PostgreSQL and a single poller. The tipping point is roughly where one processing instance is no longer enough (above a few hundred readings/s with this model), or where several independent consumers need the same stream. Our lab plant (600 readings/s) is already past it.

### Risks and consequences

- Kafka is an additional system to run and monitor.
- The number of partitions limits parallelism, so it should be planned with headroom.
- Delivery can be at-least-once (seen in Part 1, Step 3), so the detector must tolerate a repeated reading.

## Part 2, Stage C: Build the core (Kafka)

### Design decisions (predictions)

| Component | Decision | Reasoning |
|---|---|---|
| Key of `readings` | machine ID | Same key → same partition → same order per machine (R2). |
| Partitions of `readings` | 8 | About 4 instances needed (3 ms per reading, about 250/s each). 8 splits evenly over 4 and leaves room to scale. |
| Producer settings (plant) | `enable.idempotence=True`, `linger.ms=20` | No duplicates or reordering on retries. Batching keeps the load low without hurting freshness (20 ms ≪ 5 s). |
| Detector instances | 4 | About 1,000 readings/s of capacity for 600/s. |
| Consumer group | one shared group (`detector`) | Instances in one group split the partitions (Part 1, Step 3). |
| Start position | `latest` | A live detector should alert on the present and not replay old runs. The model is stateful per machine, so starting in old data would distort `missing` and `out-of-order`. Consequence: start the detectors first and do not add or remove instances during a run. |
| Key of `anomalies` | machine ID | Alerts of one machine stay in order. |
| Expected result | `missing` and `out-of-order` stay 0, load split roughly evenly (not perfectly), alerts within a few seconds | Machine key keeps order, and 60 machines hashed over 8 partitions are not perfectly even. |

### Implementation

- **`plant.py`, `KafkaSink`:** topic `readings`, `key=machine_id`, value = JSON reading, `enable.idempotence=True`, `linger.ms=20`, one retry after a `BufferError`.
- **`detector_service.py`:** defaults `--in-topic readings` and `--group detector`, `auto.offset.reset=latest`, producer with `enable.idempotence=True` and `linger.ms=10`, anomaly events keyed by `machine_id`.

### Observation

#### 1. Topics

```
docker exec -it kafka kafka-topics.sh --bootstrap-server localhost:9092 --create --topic readings --partitions 8 --replication-factor 1
docker exec -it kafka kafka-topics.sh --bootstrap-server localhost:9092 --create --topic anomalies --partitions 2 --replication-factor 1
```

`--describe --topic readings`:

```
Topic: readings TopicId: wqr99tpORmatH_NJFGvs9w PartitionCount: 8       ReplicationFactor: 1    Configs: min.insync.replicas=1
        Topic: readings Partition: 0    Leader: 1       Replicas: 1     Isr: 1  Elr:    LastKnownElr: 
        Topic: readings Partition: 1    Leader: 1       Replicas: 1     Isr: 1  Elr:    LastKnownElr: 
        Topic: readings Partition: 2    Leader: 1       Replicas: 1     Isr: 1  Elr:    LastKnownElr: 
        Topic: readings Partition: 3    Leader: 1       Replicas: 1     Isr: 1  Elr:    LastKnownElr: 
        Topic: readings Partition: 4    Leader: 1       Replicas: 1     Isr: 1  Elr:    LastKnownElr: 
        Topic: readings Partition: 5    Leader: 1       Replicas: 1     Isr: 1  Elr:    LastKnownElr: 
        Topic: readings Partition: 6    Leader: 1       Replicas: 1     Isr: 1  Elr:    LastKnownElr: 
        Topic: readings Partition: 7    Leader: 1       Replicas: 1     Isr: 1  Elr:    LastKnownElr: 
```

#### 2. Four detectors started first, then the plant

Commands (one terminal each): `python -u detector_service.py --name det-1` ... `--name det-4`, then after about 10 s: `python -u plant.py --sink kafka`.

The first detector printed a harmless warning at startup (the broker's coordinator was still loading, the producer retried by itself):

```
[det-1] readings -> anomalies (Ctrl+C to stop)
%4|1794390587.209|GETPID|rdkafka#producer-2| [thrd:main]: Failed to acquire idempotence PID from broker localhost:9092/1: Broker: Coordinator load in progress: retrying
```

Plant output:

```
run 1794390659: 60 machines x 10 Hz = 600 readings/s -> kafka (Ctrl+C to stop)
  t=    5s      3,060 readings sent (612/s)
  t=   10s      6,060 readings sent (606/s)
  t=   15s      9,060 readings sent (604/s)
  t=   20s     12,060 readings sent (603/s)
  t=   25s     15,060 readings sent (602/s)
  t=   30s     18,060 readings sent (602/s)
  t=   35s     21,060 readings sent (602/s)
  t=   40s  fault starts on M-029 (repaired in 45 s)
  t=   40s     24,060 readings sent (601/s)  faulty: M-029
  ...
  t=   80s     48,060 readings sent (601/s)  faulty: M-029
  t=   85s     51,060 readings sent (601/s)
  ...
  t=  100s  fault starts on M-026 (repaired in 45 s)
  t=  100s     60,060 readings sent (601/s)  faulty: M-026
  ...
  t=  145s     87,060 readings sent (600/s)  faulty: M-026
  t=  150s     90,060 readings sent (600/s)
  t=  155s     93,060 readings sent (600/s)
  t=  160s  fault starts on M-031 (repaired in 45 s)
  t=  160s     96,060 readings sent (600/s)  faulty: M-031
  ...
  t=  180s    108,060 readings sent (600/s)  faulty: M-031
run 1794390659: sent 108,480 readings in 181 s
```

Detector output (first and last lines of each instance, long middle parts shortened):

```
[det-1]       802 readings       0 anomalies  missing 0  out-of-order 0
[det-1]     1,825 readings       0 anomalies  missing 0  out-of-order 0
...
[det-1]    26,576 readings       0 anomalies  missing 0  out-of-order 0
[det-1] stopped after 27,120 readings, 0 anomalies

[det-2]       451 readings       0 anomalies  missing 0  out-of-order 0
...
[det-2]    25,299 readings       0 anomalies  missing 0  out-of-order 0
[det-2] stopped after 25,312 readings, 0 anomalies

[det-3]       623 readings       0 anomalies  missing 0  out-of-order 0
...
[det-3]    22,596 readings       0 anomalies  missing 0  out-of-order 0
[det-3]    23,297 readings       5 anomalies  missing 0  out-of-order 0
[det-3]    24,014 readings      56 anomalies  missing 0  out-of-order 0
[det-3]    24,711 readings     106 anomalies  missing 0  out-of-order 0
[det-3] stopped after 25,312 readings, 149 anomalies

[det-4]       637 readings       0 anomalies  missing 0  out-of-order 0
...
[det-4]     8,615 readings      28 anomalies  missing 0  out-of-order 0
...
[det-4]    29,896 readings     818 anomalies  missing 0  out-of-order 0
[det-4] stopped after 30,736 readings, 818 anomalies
```

#### 3. Consumer group during the run

Command: `docker exec -it kafka kafka-consumer-groups.sh --bootstrap-server localhost:9092 --describe --group detector`

```
GROUP           TOPIC           PARTITION  CURRENT-OFFSET  LOG-END-OFFSET  LAG             CONSUMER-ID                                HOST            CLIENT-ID
detector        readings        5          10985           11235           250             det-3-fdc42626-c5c2-4fad-b6f2-14368722ca17 /172.21.0.1     det-3
detector        readings        7          14103           14445           342             det-4-2887ed48-c4c9-4f83-b04f-57bf36866541 /172.21.0.1     det-4
detector        readings        6          12536           12840           304             det-4-2887ed48-c4c9-4f83-b04f-57bf36866541 /172.21.0.1     det-4
detector        readings        1          11032           11235           203             det-1-7740a848-0b0d-44e3-a699-e0d9cb8f1468 /172.21.0.1     det-1
detector        readings        0          12608           12840           232             det-1-7740a848-0b0d-44e3-a699-e0d9cb8f1468 /172.21.0.1     det-1
detector        readings        3          10927           11242           315             det-2-789a66d2-8b81-432c-9a11-ec8819d806fd /172.21.0.1     det-2
detector        readings        2          10927           11235           308             det-2-789a66d2-8b81-432c-9a11-ec8819d806fd /172.21.0.1     det-2
```

This output lists 7 of the 8 partitions (partition 4 is missing, probably a line lost when copying; it appears in the output after the run, see below).

#### 4. Scoreboard

Command: `python scoreboard.py`

```
Run 1794390659: started 10:50:59 (4.1 min ago), 60 machines x 10 Hz, 3 faults so far

Detection  (topic anomalies: 967 anomaly events, models ['v1'])
   #  machine   start  detected   delay
   1  M-029       40s       yes   7.9 s
   2  M-026      100s       yes   8.2 s
   3  M-031      160s       yes   5.9 s

  faults detected: 3/3, mean delay 7.3 s; false alarms (episodes on healthy machines): 0
  latency reading -> anomaly event: p50 0.12 s, p95 0.86 s, max 1.35 s
  duplicate anomaly events (same anomaly_id): 0

Detector instances  (model telemetry, this run)
  det-1          pid 22808   v1  readings    27,117  missing       0  out-of-order      0  machines   15  silent for 66 s
  det-2          pid 14068   v1  readings    25,299  missing       0  out-of-order      0  machines   14  silent for 67 s
  det-3          pid 11484   v1  readings    25,305  missing       0  out-of-order      0  machines   14  silent for 67 s
  det-4          pid 13428   v1  readings    29,823  missing       0  out-of-order      0  machines   17  silent for 72 s

Requirements (measurable part)
  [PASS] R2 complete and ordered: 0 readings missing, 0 out of order
  [PASS] R3 fresh: p95 latency < 5 s
  [PASS] detection: all faults found, at most 2 false alarms
```

#### 5. Consumer group after stopping all detectors

Command: `docker exec -it kafka kafka-consumer-groups.sh --bootstrap-server localhost:9092 --describe --group detector`

```
Consumer group 'detector' has no active members.

GROUP           TOPIC           PARTITION  CURRENT-OFFSET  LOG-END-OFFSET  LAG             CONSUMER-ID     HOST            CLIENT-ID
detector        readings        4          12656           12656           0               -               -               -
detector        readings        3          12656           12656           0               -               -               -
detector        readings        2          12656           12656           0               -               -               -
detector        readings        1          12656           12656           0               -               -               -
detector        readings        7          16272           16272           0               -               -               -
detector        readings        6          14464           14464           0               -               -               -
detector        readings        5          12656           12656           0               -               -               -
detector        readings        0          14464           14464           0               -               -               -
```

### Analysis

| Stage A (PostgreSQL, 600 readings/s) | Stage C (Kafka, 4 detectors, 600 readings/s) |
|---|---|
| Detector read 15,304 of 36,420 readings | **All 108,480 readings** processed (27,120 + 25,312 + 25,312 + 30,736) |
| 76 readings missing | **0 missing, 0 out-of-order** on every instance |
| 0 anomalies, delay 36.6 s and growing | **3 of 3 faults detected**, 0 false alarms |
| Backlog 21,040 rows | Lag 0 after the run |

- **R2 passed:** the key is the machine ID, so each machine's readings stay in one partition with one reader, in `seq` order. The scoreboard's machine counts (15 + 14 + 14 + 17 = 60) show that every machine is handled by exactly one detector.
- **R3 passed:** reading → anomaly latency is p50 0.12 s and p95 0.86 s, far below 5 s. The detection delay of 5.9 to 8.2 s after the fault start is, in my interpretation, mostly the time the model needs for the slowly rising vibration to cross its threshold. The pipeline itself adds under 1 s.
- **Parallelism and uneven load:** each detector owns 2 partitions. The partitions hold 7, 8 or 9 machines each (12,656, 14,464 and 16,272 readings, with 1,808 readings per machine), because 60 machines do not hash evenly over 8 partitions. That explains the uneven load per detector (25,312 to 30,736 readings). Each instance handled about 150 readings/s, well below its capacity of about 250 to 333/s.
- **Anomalies:** 149 (det-3) + 818 (det-4) = 967, matching the scoreboard. Only these two instances alerted because the three faulty machines happened to be assigned to them. No duplicates.
- **Lag:** during the run the lag was 200 to 340 per partition and did not grow. It is inflated by the periodic offset commits, since the committed position trails the real one. After stopping the detectors, all 8 partitions show lag 0, and the committed offsets add up to 12,656 × 5 + 14,464 × 2 + 16,272 = 108,480, exactly the number of readings the plant sent.
- **R1:** 600 readings/s needs about 2 to 3 instances (4 used). At 100,000 readings/s about 300 to 400 instances and at least as many partitions would be needed. The 8 partitions used here are enough for the lab plant but not for the target scale.
- **R4:** the scoreboard does not measure it. Design argument: an archive or a dashboard reads `readings` with its own consumer group and its own offsets, so it does not slow down the detectors and does not touch PostgreSQL.
- **Operational lesson:** with `auto.offset.reset=latest`, all detectors must be started before the plant, and instances must not be added or removed during a run, because a rebalance would move partitions to an instance without state for those machines.

**Key idea:** the Part 1 building blocks solved the problems from Stage A. Key → partition gave completeness and order, one consumer per partition gave parallelism, and offsets and lag show whether the detectors keep up.